# LeetCode 3459 — Length of Longest V-Shaped Diagonal Segment bằng Python

- Link: https://leetcode.com/problems/length-of-longest-v-shaped-diagonal-segment/
- Độ khó: **Hard**
- Chủ đề: **Quy hoạch động trên lưới** theo trạng thái `(ô, hướng chéo, đã rẽ chưa)`; bảng "đi thẳng" `S` và bảng "còn quyền rẽ" `T`; thứ tự tính theo hướng

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def lenOfVDiagonal(self, grid: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Mô hình hóa đường chéo "chữ V": bắt đầu ở ô `1`, sau đó dãy `2, 0, 2, 0, …`, đi theo một trong 4 hướng chéo, **rẽ phải 90° tối đa một lần**.
2. Nhận ra sau ô `1` chỉ cần biết **giá trị hiện tại** (0 hoặc 2) để biết giá trị kế tiếp (`2 − v`).
3. Xây hai bảng DP cho mỗi hướng `d`: `S[d]` (đi thẳng, hết quyền rẽ) và `T[d]` (đang đi hướng `d`, còn quyền rẽ).
4. Xác định **thứ tự tính** để ô kế tiếp theo hướng `d` đã có giá trị trước.
5. So sánh với DFS có nhớ (theo gợi ý đề) và vét cạn.

## 2. Đề bài & ví dụ

Lưới `n × m` gồm `0, 1, 2`. Một **đoạn chéo chữ V**:

- bắt đầu bằng ô giá trị `1`;
- các ô tiếp theo lần lượt là `2, 0, 2, 0, …`;
- đi theo một hướng chéo, và **được rẽ theo chiều kim đồng hồ 90° tối đa một lần** sang hướng chéo khác, vẫn giữ dãy giá trị.

Trả về độ dài (số ô) lớn nhất; không có ô `1` nào ⇒ `0`.

**Ví dụ 1:** → `5`; **Ví dụ 2:** → `4`; **Ví dụ 3:** → `5`; **Ví dụ 4:** `[[1]]` → `1`.

**Ràng buộc:** `1 ≤ n, m ≤ 500`.

In [1]:
from typing import List

examples = [
    ([[2,2,1,2,2],[2,0,2,2,0],[2,0,1,1,0],[1,0,2,2,2],[2,0,0,2,2]], 5),
    ([[2,2,2,2,2],[2,0,2,2,0],[2,0,1,1,0],[1,0,2,2,2],[2,0,0,2,2]], 4),
    ([[1,2,2,2,2],[2,2,2,2,0],[2,0,0,0,0],[0,0,2,2,2],[2,0,0,2,0]], 5),
    ([[1]], 1),
]
for g, exp in examples:
    print(len(g), 'x', len(g[0]), '-> kỳ vọng', exp)

5 x 5 -> kỳ vọng 5
5 x 5 -> kỳ vọng 4
5 x 5 -> kỳ vọng 5
1 x 1 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Bốn hướng chéo theo chiều kim đồng hồ

Trong hệ tọa độ (hàng tăng xuống dưới, cột tăng sang phải), quay **theo chiều kim đồng hồ** 90°:

`↘ (1, 1)` → `↙ (1, −1)` → `↖ (−1, −1)` → `↗ (−1, 1)` → `↘`.

Hướng rẽ của `DIRS[d]` là `DIRS[(d + 1) % 4]`.

In [2]:
DIRS = [(1, 1), (1, -1), (-1, -1), (-1, 1)]
NAMES = ['↘', '↙', '↖', '↗']
for d in range(4):
    print(NAMES[d], DIRS[d], '-> rẽ phải thành', NAMES[(d + 1) % 4], DIRS[(d + 1) % 4])

↘ (1, 1) -> rẽ phải thành ↙ (1, -1)
↙ (1, -1) -> rẽ phải thành ↖ (-1, -1)
↖ (-1, -1) -> rẽ phải thành ↗ (-1, 1)
↗ (-1, 1) -> rẽ phải thành ↘ (1, 1)


Kiểm tra bằng ví dụ 1: `(0,2) → (1,3) → (2,4)` là hướng `↘`, sau đó `(3,3) → (4,2)` là `↙` — đúng là rẽ phải.

### 3.2. Thứ tự duyệt để "ô kế tiếp" được tính trước

Nếu `S[d][i][j]` phụ thuộc `S[d][i + di][j + dj]` thì với `di = 1` duyệt hàng **từ dưới lên**, `di = −1` duyệt **từ trên xuống**; tương tự cho cột.

In [3]:
def order(n, m, d):
    di, dj = DIRS[d]
    rows = range(n - 1, -1, -1) if di == 1 else range(n)
    cols = range(m - 1, -1, -1) if dj == 1 else range(m)
    return rows, cols

for d in range(4):
    r, c = order(3, 3, d)
    print(NAMES[d], 'hàng', list(r), 'cột', list(c))

↘ hàng [2, 1, 0] cột [2, 1, 0]
↙ hàng [2, 1, 0] cột [0, 1, 2]
↖ hàng [0, 1, 2] cột [0, 1, 2]
↗ hàng [0, 1, 2] cột [2, 1, 0]


### 3.3. Dãy xen kẽ sau ô `1`

Ô ngay sau `1` phải là `2`. Từ một ô giá trị `v ∈ {0, 2}`, ô kế tiếp phải là `2 − v`. Vậy với các ô `0/2`, độ dài chuỗi xen kẽ chỉ phụ thuộc **ô bắt đầu** và **hướng** — không phụ thuộc đã đi bao xa.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 3: ô `(0,0) = 1`, đi `↘`: `(1,1)=2, (2,2)=0, (3,3)=2, (4,4)=0` ⇒ dãy `1,2,0,2,0` đúng ⇒ độ dài 5, không cần rẽ.

Ví dụ 2: ô `(2,3) = 1`, đi `↙`: `(3,2) = 2`; rẽ phải thành `↖`: `(2,1) = 0`, `(1,0) = 2` ⇒ độ dài 4.

Quan sát: một đoạn chữ V = ô `1` + **phần đầu** (hướng `d`) + **phần sau** (hướng `d'` sau khi rẽ). Phần sau là chuỗi xen kẽ đi thẳng — tính sẵn được.

In [4]:
g = examples[1][0]
print([g[i][j] for i, j in [(2, 3), (3, 2), (2, 1), (1, 0)]])

[1, 2, 0, 2]


## 5. Cách 1 — Brute Force: DFS mọi đường đi từ mỗi ô `1`

Từ mỗi ô `1`, mỗi hướng: tại mỗi bước có thể đi thẳng hoặc (nếu chưa rẽ) rẽ phải. Số đường từ một điểm xuất phát là `O(L²)` (chọn chỗ rẽ), mỗi đường dài `O(L)`.

In [5]:
def lvd_brute(g):
    n, m = len(g), len(g[0])
    best = 0

    def walk(i, j, d, turned, expect, length):
        nonlocal best
        best = max(best, length)
        for nd in ([d] if turned else [d, (d + 1) % 4]):
            a, b = i + DIRS[nd][0], j + DIRS[nd][1]
            if 0 <= a < n and 0 <= b < m and g[a][b] == expect:
                walk(a, b, nd, turned or nd != d, 2 - expect, length + 1)

    for i in range(n):
        for j in range(m):
            if g[i][j] == 1:
                for d in range(4):
                    walk(i, j, d, False, 2, 1)
    return best

for g, exp in examples:
    print(lvd_brute(g), '| kỳ vọng', exp)

5 | kỳ vọng 5
4 | kỳ vọng 4
5 | kỳ vọng 5
1 | kỳ vọng 1


**Độ phức tạp:** với mỗi ô `1` và hướng: `O(L²)` đường (L ≤ `min(n, m)`) — tệ nhất `O(n m L²)`, chỉ dùng cho lưới nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

Với mỗi hướng `d` và mỗi ô `c` có giá trị `v ∈ {0, 2}` (coi `c` đã là một phần hợp lệ của đoạn):

- `S[d][c]` = số ô dài nhất đi **thẳng** theo `d` bắt đầu từ `c` giữ xen kẽ: `1 + S[d][c + d]` nếu `c + d` trong lưới và có giá trị `2 − v`, ngược lại `1`.
- `T[d][c]` = số ô dài nhất bắt đầu từ `c`, đang đi hướng `d`, **còn quyền rẽ**: `1 + max( T[d][c + d], S[d'][c + d'] )` (mỗi nhánh chỉ tính nếu ô tới có giá trị `2 − v`), với `d' = (d + 1) mod 4`.

Đáp án: với mỗi ô `1` và hướng `d`, nếu ô `c + d` có giá trị `2` thì ứng viên `1 + T[d][c + d]`; tối thiểu `1` nếu có ô `1`.

(Rẽ ngay tại ô `1` tương đương bắt đầu bằng hướng `d'` — đã được xét.)

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DFS có nhớ theo trạng thái `(i, j, d, đã rẽ)` (theo gợi ý)

In [6]:
import sys
from functools import lru_cache

def lvd_memo(g):
    n, m = len(g), len(g[0])
    sys.setrecursionlimit(max(10000, 4 * (n + m) + 100))

    @lru_cache(maxsize=None)
    def f(i, j, d, turned):
        # (i, j) đã thuộc đoạn, giá trị g[i][j] ∈ {0, 2} hoặc 1 ở ô đầu; trả về số ô từ (i, j) trở đi
        v = g[i][j]
        need = 2 if v == 1 else 2 - v
        best = 0
        for nd in ([d] if turned else [d, (d + 1) % 4]):
            a, b = i + DIRS[nd][0], j + DIRS[nd][1]
            if 0 <= a < n and 0 <= b < m and g[a][b] == need:
                best = max(best, f(a, b, nd, turned or nd != d))
        return 1 + best

    ans = 0
    for i in range(n):
        for j in range(m):
            if g[i][j] == 1:
                for d in range(4):
                    ans = max(ans, f(i, j, d, False))
    f.cache_clear()
    return ans

for g, exp in examples:
    print(lvd_memo(g), '| kỳ vọng', exp)

5 | kỳ vọng 5
4 | kỳ vọng 4
5 | kỳ vọng 5
1 | kỳ vọng 1


Trạng thái `n · m · 4 · 2 = 2·10^6` ở cỡ lớn nhất — đúng nhưng `lru_cache` + đệ quy trong Python khá chậm và tốn bộ nhớ.

### 7.2. Cách 3 — Bảng `S` và `T` tính lặp (lời giải chính)

In [7]:
def lvd_tables(g):
    n, m = len(g), len(g[0])
    S = [[[0] * m for _ in range(n)] for _ in range(4)]
    for d, (di, dj) in enumerate(DIRS):
        rows, cols = order(n, m, d)
        Sd = S[d]
        for i in rows:
            gi = g[i]
            for j in cols:
                v = gi[j]
                if v == 1:
                    continue
                a, b = i + di, j + dj
                Sd[i][j] = 1 + (Sd[a][b] if 0 <= a < n and 0 <= b < m and g[a][b] == 2 - v else 0)
    T = [[[0] * m for _ in range(n)] for _ in range(4)]
    for d, (di, dj) in enumerate(DIRS):
        d2 = (d + 1) % 4
        ei, ej = DIRS[d2]
        S2, Td = S[d2], T[d]
        rows, cols = order(n, m, d)
        for i in rows:
            for j in cols:
                v = g[i][j]
                if v == 1:
                    continue
                best = 0
                a, b = i + di, j + dj
                if 0 <= a < n and 0 <= b < m and g[a][b] == 2 - v:
                    best = Td[a][b]
                a, b = i + ei, j + ej
                if 0 <= a < n and 0 <= b < m and g[a][b] == 2 - v and S2[a][b] > best:
                    best = S2[a][b]
                Td[i][j] = 1 + best
    ans = 0
    for i in range(n):
        for j in range(m):
            if g[i][j] == 1:
                cur = 1
                for d, (di, dj) in enumerate(DIRS):
                    a, b = i + di, j + dj
                    if 0 <= a < n and 0 <= b < m and g[a][b] == 2:
                        cur = max(cur, 1 + T[d][a][b])
                ans = max(ans, cur)
    return ans

for g, exp in examples:
    print(lvd_tables(g), '| kỳ vọng', exp)

5 | kỳ vọng 5
4 | kỳ vọng 4
5 | kỳ vọng 5
1 | kỳ vọng 1


### 7.3. Cách 4 — Đi từ mỗi ô `1`, thử mọi điểm rẽ, dùng bảng `S`

Từ ô `1`, đi thẳng theo `d` qua các ô hợp lệ; tại mỗi ô `p` trên đường (kể cả ô `1`) thử rẽ: `độ dài phần đầu + S[d'][p + d']` nếu ô `p + d'` mang giá trị đúng. Đúng, đơn giản, nhưng tốn `O(L)` cho mỗi (ô `1`, hướng) ⇒ tệ nhất `O(n m L)` khi có nhiều ô `1`.

In [8]:
def lvd_walk(g):
    n, m = len(g), len(g[0])
    S = [[[0] * m for _ in range(n)] for _ in range(4)]
    for d, (di, dj) in enumerate(DIRS):
        rows, cols = order(n, m, d)
        for i in rows:
            for j in cols:
                v = g[i][j]
                if v == 1:
                    continue
                a, b = i + di, j + dj
                S[d][i][j] = 1 + (S[d][a][b] if 0 <= a < n and 0 <= b < m and g[a][b] == 2 - v else 0)
    ans = 0
    for i in range(n):
        for j in range(m):
            if g[i][j] != 1:
                continue
            ans = max(ans, 1)
            for d, (di, dj) in enumerate(DIRS):
                d2 = (d + 1) % 4
                ei, ej = DIRS[d2]
                pi, pj, length, need = i, j, 1, 2
                while True:
                    a, b = pi + ei, pj + ej              # thử rẽ tại p
                    if 0 <= a < n and 0 <= b < m and g[a][b] == need:
                        ans = max(ans, length + S[d2][a][b])
                    a, b = pi + di, pj + dj              # đi thẳng
                    if not (0 <= a < n and 0 <= b < m and g[a][b] == need):
                        break
                    pi, pj, length, need = a, b, length + 1, 2 - need
                    ans = max(ans, length)
    return ans

for g, exp in examples:
    print(lvd_walk(g), '| kỳ vọng', exp)

5 | kỳ vọng 5
4 | kỳ vọng 4
5 | kỳ vọng 5
1 | kỳ vọng 1


## 8. TRACE từng bước

Trace cách 4 trên ví dụ 1 cho ô `1` tại `(0, 2)`, hướng `↘`: in từng ô đi thẳng và giá trị nhận được nếu rẽ ở đó.

In [9]:
def trace_walk(g, i, j, d):
    n, m = len(g), len(g[0])
    S = {}
    def straight(a, b, dd):
        # độ dài chuỗi xen kẽ đi thẳng (tính trực tiếp để trace dễ đọc)
        L, v = 1, g[a][b]
        while True:
            x, y = a + DIRS[dd][0] * L, b + DIRS[dd][1] * L
            if not (0 <= x < n and 0 <= y < m and g[x][y] == (2 - v if L % 2 else v)):
                return L
            L += 1
    d2 = (d + 1) % 4
    pi, pj, length, need = i, j, 1, 2
    print(f"bắt đầu tại ({i},{j}) giá trị 1, hướng {NAMES[d]}, rẽ sang {NAMES[d2]}")
    best = 1
    while True:
        a, b = pi + DIRS[d2][0], pj + DIRS[d2][1]
        if 0 <= a < n and 0 <= b < m and g[a][b] == need:
            s = straight(a, b, d2)
            best = max(best, length + s)
            print(f"  tại ({pi},{pj}) (đã {length} ô): rẽ được, phần sau dài {s} -> tổng {length + s}")
        else:
            print(f"  tại ({pi},{pj}) (đã {length} ô): không rẽ được (cần {need})")
        a, b = pi + DIRS[d][0], pj + DIRS[d][1]
        if not (0 <= a < n and 0 <= b < m and g[a][b] == need):
            print(f"  không đi thẳng tiếp được -> dừng; tốt nhất {max(best, length)}")
            return max(best, length)
        pi, pj, length, need = a, b, length + 1, 2 - need

trace_walk(examples[0][0], 0, 2, 0)

bắt đầu tại (0,2) giá trị 1, hướng ↘, rẽ sang ↙
  tại (0,2) (đã 1 ô): không rẽ được (cần 2)
  tại (1,3) (đã 2 ô): không rẽ được (cần 0)
  tại (2,4) (đã 3 ô): rẽ được, phần sau dài 2 -> tổng 5
  không đi thẳng tiếp được -> dừng; tốt nhất 5


5

## 9. Code nộp LeetCode

In [10]:
class Solution:
    def lenOfVDiagonal(self, grid: List[List[int]]) -> int:
        n, m = len(grid), len(grid[0])
        dirs = [(1, 1), (1, -1), (-1, -1), (-1, 1)]

        def ranges(di, dj):
            rows = range(n - 1, -1, -1) if di == 1 else range(n)
            cols = range(m - 1, -1, -1) if dj == 1 else range(m)
            return rows, cols

        S = [[[0] * m for _ in range(n)] for _ in range(4)]
        for d, (di, dj) in enumerate(dirs):
            rows, cols = ranges(di, dj)
            Sd = S[d]
            for i in rows:
                for j in cols:
                    v = grid[i][j]
                    if v == 1:
                        continue
                    a, b = i + di, j + dj
                    if 0 <= a < n and 0 <= b < m and grid[a][b] == 2 - v:
                        Sd[i][j] = 1 + Sd[a][b]
                    else:
                        Sd[i][j] = 1
        T = [[[0] * m for _ in range(n)] for _ in range(4)]
        for d, (di, dj) in enumerate(dirs):
            ei, ej = dirs[(d + 1) % 4]
            S2, Td = S[(d + 1) % 4], T[d]
            rows, cols = ranges(di, dj)
            for i in rows:
                for j in cols:
                    v = grid[i][j]
                    if v == 1:
                        continue
                    best = 0
                    a, b = i + di, j + dj
                    if 0 <= a < n and 0 <= b < m and grid[a][b] == 2 - v:
                        best = Td[a][b]
                    a, b = i + ei, j + ej
                    if 0 <= a < n and 0 <= b < m and grid[a][b] == 2 - v and S2[a][b] > best:
                        best = S2[a][b]
                    Td[i][j] = 1 + best
        ans = 0
        for i in range(n):
            for j in range(m):
                if grid[i][j] == 1:
                    cur = 1
                    for d, (di, dj) in enumerate(dirs):
                        a, b = i + di, j + dj
                        if 0 <= a < n and 0 <= b < m and grid[a][b] == 2 and T[d][a][b] + 1 > cur:
                            cur = T[d][a][b] + 1
                    if cur > ans:
                        ans = cur
        return ans

for g, exp in examples:
    got = Solution().lenOfVDiagonal(g)
    print(got, 'OK' if got == exp else 'SAI')

5 OK
4 OK
5 OK
1 OK


## 10. Vì sao thuật toán đúng

1. **Cấu trúc đoạn:** đoạn chữ V = ô `1`, rồi một số ô theo hướng `d` (≥ 0), rồi (nếu rẽ) một số ô theo `d' = d + 1`. Mỗi ô sau ô `1` có giá trị xác định duy nhất bởi ô trước (`2` sau `1`; `2 − v` sau `v`).
2. **`S` đúng:** chuỗi đi thẳng xen kẽ dài nhất từ `c` theo `d` là `c` cộng chuỗi từ `c + d` nếu `c + d` hợp lệ — quy nạp theo thứ tự tính (ô `c + d` luôn tính trước nhờ thứ tự duyệt).
3. **`T` đúng:** từ `c` còn quyền rẽ, bước kế tiếp hoặc đi thẳng (vẫn còn quyền rẽ ⇒ `T[d]`) hoặc rẽ (hết quyền ⇒ `S[d']`); hoặc dừng tại `c` (độ dài 1). Lấy max.
4. **Đáp án:** mọi đoạn có độ dài ≥ 2 bắt đầu bằng ô `1` rồi ô `2` theo hướng ban đầu `d` (rẽ ngay tại ô `1` coi như bắt đầu bằng `d'`); đoạn độ dài 1 là chỉ ô `1`.
5. **Rẽ đúng chiều:** chỉ cho phép `d → (d + 1) mod 4` (theo chiều kim đồng hồ theo thứ tự `DIRS`).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. DFS mọi đường | `O(#1 · 4 · L²)` | `O(L)` đệ quy |
| 2. DFS có nhớ | `O(n m · 8)` trạng thái | `O(n m · 8)` |
| 3. Bảng `S`, `T` | `O(n m · 8)` | `O(n m · 8)` |
| 4. Đi từ mỗi ô `1` | `O(n m + #1 · 4 · L)` | `O(n m · 4)` |

Với `n = m = 500`: `n m · 8 = 2·10^6` bước cập nhật — khoảng 1–2 giây trong Python thuần.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'memo': lvd_memo, 'tables': lvd_tables, 'walk': lvd_walk,
         'submit': lambda g: Solution().lenOfVDiagonal(g)}
edge = [
    ([[0]], 0), ([[2, 2], [2, 2]], 0),
    ([[1, 2], [2, 1]], 1),                     # ô 2 chỉ kề ngang/dọc với ô 1, không kề chéo
    ([[1, 0], [0, 2]], 2),                     # đi ↘ tới (1,1)=2
    ([[2, 1, 2]], 1),                          # một hàng: không có đường chéo
    ([[1, 0, 0], [0, 2, 0], [0, 0, 0]], 3),    # (0,0)->(1,1)=2 ->(2,2)=0
    ([[1, 0, 0], [0, 2, 0], [0, 0, 2]], 3),    # (2,2) sai giá trị, nhưng rẽ ↙ tới (2,0)=0
]
for g, exp in examples + edge:
    assert lvd_brute(g) == exp, (g, lvd_brute(g))
    for name, f in impls.items():
        assert f(g) == exp, (name, g)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [12]:
import random
rng = random.Random(3459)
cnt = 0
for _ in range(1500):
    n, m = rng.randint(1, 7), rng.randint(1, 7)
    g = [[rng.choice([0, 1, 2, 2, 0]) for _ in range(m)] for _ in range(n)]
    exp = lvd_brute(g)
    for name, f in impls.items():
        assert f(g) == exp, (name, g)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'lưới ngẫu nhiên.')

Khớp vét cạn trên 1500 lưới ngẫu nhiên.


### 12.3. Ca lớn `500 × 500`

In [13]:
import time
N = 500
diag = [[(2 if i % 2 else 0) for j in range(N)] for i in range(N)]
diag[0][0] = 1                                    # đường chéo chính dài 500
zigzag = [[rng.choice([0, 2]) for _ in range(N)] for _ in range(N)]
for _ in range(2000):
    zigzag[rng.randrange(N)][rng.randrange(N)] = 1
ones = [[1 if (i + j) % 3 == 0 else (2 if (i + j) % 3 == 1 else 0) for j in range(N)] for i in range(N)]
for name, g in [('đường chéo dài', diag), ('ngẫu nhiên 0/2 + 2000 ô 1', zigzag), ('nhiều ô 1', ones)]:
    t = time.time(); a = Solution().lenOfVDiagonal(g); t1 = time.time() - t
    t = time.time(); b = lvd_walk(g); t2 = time.time() - t
    assert a == b
    print(f"{name:26s}: {a:4d} | bảng S,T {t1:.2f}s | đi từ ô 1 {t2:.2f}s")

g = [[2 if i % 2 else (1 if (i * 7 + j) % 5 == 0 else 0) for j in range(150)] for i in range(150)]   # nhiều ô 1, chuỗi 0/2 dài
t = time.time(); r = lvd_memo(g); t3 = time.time() - t
assert r == Solution().lenOfVDiagonal(g)
print(f"DFS có nhớ ở 150 x 150: {t3:.2f}s (cỡ 500 x 500 sẽ chậm hơn ~11 lần và tốn nhiều bộ nhớ)")

đường chéo dài            :  500 | bảng S,T 0.47s | đi từ ô 1 0.16s


ngẫu nhiên 0/2 + 2000 ô 1 :   17 | bảng S,T 0.41s | đi từ ô 1 0.15s


nhiều ô 1                 :    3 | bảng S,T 0.32s | đi từ ô 1 0.26s
DFS có nhớ ở 150 x 150: 0.13s (cỡ 500 x 500 sẽ chậm hơn ~11 lần và tốn nhiều bộ nhớ)


## 13. Các lỗi người mới hay gặp

Hàm tìm phản ví dụ (seed cố định):

In [14]:
def find_counterexample(wrong, seed=0, tries=20000):
    rng = random.Random(seed)
    best = None
    for _ in range(tries):
        n, m = rng.randint(1, 5), rng.randint(1, 5)
        g = [[rng.choice([0, 1, 2, 2, 0]) for _ in range(m)] for _ in range(n)]
        if wrong(g) != lvd_brute(g) and (best is None or n * m < len(best) * len(best[0])):
            best = g
    return best

def make_brute(turn_dirs, max_turns=1, restart_after_turn=False):
    def solve(g):
        n, m = len(g), len(g[0])
        best = 0
        def walk(i, j, d, turns, expect, length):
            nonlocal best
            best = max(best, length)
            opts = [(d, turns)] + ([((d + k) % 4, turns + 1) for k in turn_dirs] if turns < max_turns else [])
            for nd, nt in opts:
                a, b = i + DIRS[nd][0], j + DIRS[nd][1]
                e = 2 if (restart_after_turn and nd != d) else expect
                if 0 <= a < n and 0 <= b < m and g[a][b] == e:
                    walk(a, b, nd, nt, 2 - e, length + 1)
        for i in range(n):
            for j in range(m):
                if g[i][j] == 1:
                    for d in range(4):
                        walk(i, j, d, 0, 2, 1)
        return best
    return solve

### Lỗi 1 — Cho phép rẽ cả hai chiều (trái hoặc phải)

In [15]:
wrong_both = make_brute(turn_dirs=[1, 3])
ce = find_counterexample(wrong_both, seed=1)
print('phản ví dụ:', ce)
print('sai :', wrong_both(ce), '| đúng:', lvd_brute(ce))

phản ví dụ: [[0, 2, 0], [0, 0, 1]]
sai : 3 | đúng: 2


**Vì sao sai:** đề chỉ cho rẽ **theo chiều kim đồng hồ**; rẽ trái tạo đường dài hơn không hợp lệ.

### Lỗi 2 — Sau khi rẽ, bắt đầu lại dãy từ `2`

In [16]:
wrong_restart = make_brute(turn_dirs=[1], restart_after_turn=True)
ce = find_counterexample(wrong_restart, seed=2)
print('phản ví dụ:', ce)
print('sai :', wrong_restart(ce), '| đúng:', lvd_brute(ce))

phản ví dụ: [[1, 2], [1, 2], [2, 0]]
sai : 3 | đúng: 2


**Vì sao sai:** dãy `2, 0, 2, 0, …` **tiếp tục** qua chỗ rẽ ("while maintaining the sequence").

### Lỗi 3 — Cho phép rẽ nhiều lần

In [17]:
wrong_many = make_brute(turn_dirs=[1], max_turns=3)
ce = find_counterexample(wrong_many, seed=3)
print('phản ví dụ:', ce)
print('sai :', wrong_many(ce), '| đúng:', lvd_brute(ce))

phản ví dụ: [[0, 1, 2], [2, 2, 2], [1, 0, 1]]
sai : 4 | đúng: 3


**Vì sao sai:** tối đa **một** lần rẽ; thêm lần rẽ thứ hai có thể nối thêm ô.

### Lỗi 4 — Thiếu trường hợp đoạn chỉ gồm ô `1`

In [18]:
def wrong_min_two(g):
    r = Solution().lenOfVDiagonal(g)
    n, m = len(g), len(g[0])
    best = 0
    for i in range(n):
        for j in range(m):
            if g[i][j] == 1:
                for di, dj in DIRS:
                    a, b = i + di, j + dj
                    if 0 <= a < n and 0 <= b < m and g[a][b] == 2:
                        best = max(best, r)
    return best                                          # SAI: = 0 nếu không ô 1 nào có ô 2 kề chéo

print('sai :', wrong_min_two([[1]]), wrong_min_two([[2, 1, 2]]), '| đúng:', Solution().lenOfVDiagonal([[1]]), Solution().lenOfVDiagonal([[2, 1, 2]]))

sai : 0 0 | đúng: 1 1


**Vì sao sai:** ví dụ 4 của đề: chỉ ô `1` đã là đoạn độ dài 1. Phải khởi tạo `cur = 1` cho mỗi ô `1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đường đi theo hướng cố định, được đổi hướng tối đa `t` lần" ⇒ DP `(ô, hướng, số lần đổi còn lại)`.
- Giá trị ô kế tiếp xác định bởi ô hiện tại ⇒ không cần lưu "đã đi bao xa".
- Lưới lớn trong Python ⇒ tính lặp theo thứ tự hướng thay vì đệ quy có nhớ.
- Đoạn có điểm đầu đặc biệt (`1`) ⇒ tính DP cho phần "đuôi" rồi ghép.

## 15. Biến thể của bài

1. Cho rẽ tối đa `t` lần ⇒ bảng `T_t[d]` cho `t = 0..t`.
2. Cho rẽ cả trái lẫn phải ⇒ thêm nhánh `S[(d + 3) % 4]`.
3. Dãy giá trị khác (ví dụ chu kỳ `2, 0, 0`) ⇒ trạng thái thêm vị trí trong chu kỳ.
4. Đường đi theo 4 hướng ngang/dọc thay vì chéo ⇒ cùng khung DP.

In [19]:
# Biến thể 1: tối đa t lần rẽ (phải) — DP theo số lần rẽ còn lại
def lvd_t_turns(g, t):
    n, m = len(g), len(g[0])
    prev = None                                         # bảng với số lần rẽ còn lại = k - 1
    for k in range(t + 1):
        cur = [[[0] * m for _ in range(n)] for _ in range(4)]
        for d, (di, dj) in enumerate(DIRS):
            d2 = (d + 1) % 4
            rows, cols = order(n, m, d)
            for i in rows:
                for j in cols:
                    v = g[i][j]
                    if v == 1:
                        continue
                    best = 0
                    a, b = i + di, j + dj
                    if 0 <= a < n and 0 <= b < m and g[a][b] == 2 - v:
                        best = cur[d][a][b]
                    if prev is not None:
                        a, b = i + DIRS[d2][0], j + DIRS[d2][1]
                        if 0 <= a < n and 0 <= b < m and g[a][b] == 2 - v:
                            best = max(best, prev[d2][a][b])
                    cur[d][i][j] = 1 + best
        prev = cur
    ans = 0
    for i in range(n):
        for j in range(m):
            if g[i][j] == 1:
                ans = max(ans, 1)
                for d, (di, dj) in enumerate(DIRS):
                    a, b = i + di, j + dj
                    if 0 <= a < n and 0 <= b < m and g[a][b] == 2:
                        ans = max(ans, 1 + prev[d][a][b])
    return ans

rng = random.Random(15)
for _ in range(500):
    n, m = rng.randint(1, 6), rng.randint(1, 6)
    g = [[rng.choice([0, 1, 2, 2, 0]) for _ in range(m)] for _ in range(n)]
    for t in range(3):
        assert lvd_t_turns(g, t) == make_brute([1], max_turns=t)(g)
    assert lvd_t_turns(g, 1) == Solution().lenOfVDiagonal(g)
print('Biến thể t lần rẽ khớp vét cạn với t = 0, 1, 2.')

Biến thể t lần rẽ khớp vét cạn với t = 0, 1, 2.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | DFS mọi đường | `O(#1 · L²)` | Oracle |
| 2 | DFS có nhớ | `O(8 n m)` | Đúng gợi ý, chậm/tốn bộ nhớ trong Python |
| 3 | Bảng `S`, `T` lặp | `O(8 n m)` | **Nộp bài** |
| 4 | Đi từ ô `1` + bảng `S` | `O(n m + #1 · L)` | Dễ hiểu, phụ thuộc số ô `1` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[[1, 2], [2, 0]]`.

**Bài 2 (tính tay):** lưới `3 × 3` toàn `2` trừ tâm là `1`.

**Bài 3 (điền TODO):** viết `straight_only(grid)` — đoạn dài nhất **không rẽ**.

**Bài 4 (tự viết lại):** trả về cả **tọa độ** đoạn dài nhất (một đoạn bất kỳ).

**Bài 5 (mini challenge):** biến thể 2 — cho rẽ trái hoặc phải (vẫn tối đa một lần).

In [20]:
def straight_only_todo(grid):
    # TODO
    raise NotImplementedError

def best_path_todo(grid):
    # TODO
    raise NotImplementedError

def any_turn_todo(grid):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ô `1` ở `(0,0)`. (2) Đi `↘` tới `(1,1) = 0` — sai giá trị. (3) Các ô `2` kề ngang/dọc không tính.

**Bài 2.** (1) Từ tâm đi 4 hướng chéo đều gặp `2`. (2) Sau `2` cần `0` — không có. (3) Độ dài 2.

**Bài 3.** (1) Chỉ cần bảng `S`. (2) Ghép như cách 3 nhưng thay `T` bằng `S`. (3) Đối chiếu `lvd_t_turns(g, 0)`.

**Bài 4.** (1) Lưu lựa chọn tốt nhất (đi thẳng hay rẽ) trong `T`. (2) Dựng lại từ ô `1`. (3) Kiểm tra đường dựng được là hợp lệ và có độ dài bằng đáp án.

**Bài 5.** (1) `T[d][c] = 1 + max(T[d][c+d], S[d+1][…], S[d+3][…])`. (2) Đối chiếu `make_brute([1, 3])`. (3) Chú ý rẽ ngay tại ô `1`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
a1 = Solution().lenOfVDiagonal([[1, 2], [2, 0]])
g2 = [[2, 2, 2], [2, 1, 2], [2, 2, 2]]
a2 = Solution().lenOfVDiagonal(g2)
assert a1 == lvd_brute([[1, 2], [2, 0]]) == 1 and a2 == lvd_brute(g2) == 2
print('Bài 1:', a1, '| Bài 2:', a2)

straight_only = lambda g: lvd_t_turns(g, 0)
rng = random.Random(3)
for _ in range(300):
    n, m = rng.randint(1, 6), rng.randint(1, 6)
    g = [[rng.choice([0, 1, 2, 2, 0]) for _ in range(m)] for _ in range(n)]
    assert straight_only(g) == make_brute([1], max_turns=0)(g)
print('Bài 3 khớp vét cạn; ví dụ 1 không rẽ:', straight_only(examples[0][0]))

Bài 1: 1 | Bài 2: 2
Bài 3 khớp vét cạn; ví dụ 1 không rẽ: 3


In [22]:
# Bài 4 — dựng lại đường đi
def best_path(grid):
    n, m = len(grid), len(grid[0])
    best_len, best_path_ = 0, []
    def straight_cells(i, j, d, need):
        cells = []
        while 0 <= i < n and 0 <= j < m and grid[i][j] == need:
            cells.append((i, j)); need = 2 - need
            i += DIRS[d][0]; j += DIRS[d][1]
        return cells
    for i in range(n):
        for j in range(m):
            if grid[i][j] != 1:
                continue
            if best_len < 1:
                best_len, best_path_ = 1, [(i, j)]
            for d in range(4):
                d2 = (d + 1) % 4
                first = [(i, j)] + straight_cells(i + DIRS[d][0], j + DIRS[d][1], d, 2)
                for k in range(len(first)):          # rẽ sau ô first[k]
                    pi, pj = first[k]
                    need = 2 if k == 0 else 2 - grid[pi][pj]
                    tail = straight_cells(pi + DIRS[d2][0], pj + DIRS[d2][1], d2, need)
                    path = first[:k + 1] + tail
                    if len(path) > best_len:
                        best_len, best_path_ = len(path), path
    return best_len, best_path_

def valid_v(grid, path):
    if not path or grid[path[0][0]][path[0][1]] != 1:
        return False
    vals = [grid[i][j] for i, j in path[1:]]
    if any(v != (2 if k % 2 == 0 else 0) for k, v in enumerate(vals)):
        return False
    steps = [(b[0] - a[0], b[1] - a[1]) for a, b in zip(path, path[1:])]
    if any(s not in DIRS for s in steps):
        return False
    changes = [(DIRS.index(s), DIRS.index(t)) for s, t in zip(steps, steps[1:]) if s != t]
    return len(changes) <= 1 and all(t == (s + 1) % 4 for s, t in changes)

rng = random.Random(4)
for _ in range(300):
    n, m = rng.randint(1, 6), rng.randint(1, 6)
    g = [[rng.choice([0, 1, 2, 2, 0]) for _ in range(m)] for _ in range(n)]
    L, p = best_path(g)
    assert L == Solution().lenOfVDiagonal(g) and (L == 0 or valid_v(g, p))
print('Bài 4: đường dựng lại hợp lệ; ví dụ 1:', best_path(examples[0][0]))

Bài 4: đường dựng lại hợp lệ; ví dụ 1: (5, [(0, 2), (1, 3), (2, 4), (3, 3), (4, 2)])


In [23]:
# Bài 5 — rẽ trái hoặc phải
def any_turn(grid):
    n, m = len(grid), len(grid[0])
    S = [[[0] * m for _ in range(n)] for _ in range(4)]
    for d, (di, dj) in enumerate(DIRS):
        rows, cols = order(n, m, d)
        for i in rows:
            for j in cols:
                v = grid[i][j]
                if v == 1:
                    continue
                a, b = i + di, j + dj
                S[d][i][j] = 1 + (S[d][a][b] if 0 <= a < n and 0 <= b < m and grid[a][b] == 2 - v else 0)
    T = [[[0] * m for _ in range(n)] for _ in range(4)]
    for d, (di, dj) in enumerate(DIRS):
        rows, cols = order(n, m, d)
        for i in rows:
            for j in cols:
                v = grid[i][j]
                if v == 1:
                    continue
                best = 0
                a, b = i + di, j + dj
                if 0 <= a < n and 0 <= b < m and grid[a][b] == 2 - v:
                    best = T[d][a][b]
                for d2 in ((d + 1) % 4, (d + 3) % 4):
                    a, b = i + DIRS[d2][0], j + DIRS[d2][1]
                    if 0 <= a < n and 0 <= b < m and grid[a][b] == 2 - v:
                        best = max(best, S[d2][a][b])
                T[d][i][j] = 1 + best
    ans = 0
    for i in range(n):
        for j in range(m):
            if grid[i][j] == 1:
                ans = max(ans, 1)
                for d, (di, dj) in enumerate(DIRS):
                    a, b = i + di, j + dj
                    if 0 <= a < n and 0 <= b < m and grid[a][b] == 2:
                        ans = max(ans, 1 + T[d][a][b])
    return ans

rng = random.Random(5)
for _ in range(500):
    n, m = rng.randint(1, 6), rng.randint(1, 6)
    g = [[rng.choice([0, 1, 2, 2, 0]) for _ in range(m)] for _ in range(n)]
    assert any_turn(g) == make_brute([1, 3])(g)
print('Bài 5 khớp vét cạn; ví dụ 1:', any_turn(examples[0][0]))

Bài 5 khớp vét cạn; ví dụ 1: 5


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
DIRS = [(1,1), (1,-1), (-1,-1), (-1,1)]       # theo chiều kim đồng hồ
S[d][c] = 1 + S[d][c+d]           nếu g[c+d] == 2 - g[c]   (đi thẳng)
T[d][c] = 1 + max(T[d][c+d], S[d+1][c+(d+1)])  (mỗi nhánh cần giá trị 2 - g[c])
đáp án = max over ô 1: max(1, 1 + T[d][c+d] nếu g[c+d] == 2)
thứ tự duyệt: hàng ngược nếu di = 1, cột ngược nếu dj = 1
```

**Checklist:**

- [ ] Rẽ đúng chiều kim đồng hồ (`d → d + 1` trong thứ tự `DIRS`)?
- [ ] Dãy tiếp tục qua chỗ rẽ, không bắt đầu lại?
- [ ] Tối đa một lần rẽ?
- [ ] Ô `1` một mình là độ dài 1; không có ô `1` ⇒ 0?
- [ ] Không đệ quy sâu / `lru_cache` khổng lồ ở lưới 500 × 500?